# Siemens Deep Dive - Electrolyzer Patent Analysis

### Focused analysis of Siemens and its subsidiaries/variations in the electrolyzer patent landscape

Based on the general electrolyzer patent analysis (Notebook 12), this notebook:

**Key Features:**
- **Name Harmonization**: Maps all Siemens name variations to corporate groups (Siemens AG, Siemens Energy, Siemens Gamesa, Joint Ventures)
- **Technology Profile**: IPC/CPC classification analysis to understand Siemens' technology focus
- **Timeline Analysis**: Evolution of Siemens' electrolyzer patent activity over time
- **Competitive Context**: Siemens vs. top applicants comparison
- **Geographic Strategy**: Filing countries and patent office distribution

**Data Sources:**
- Electrolyzer Enhanced Final Dataset 2025
- PATSTAT PROD (TLS201, TLS206, TLS207, TLS209, TLS224, TLS211)

In [1]:
# Import libraries
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN, TLS206_PERSON, TLS207_PERS_APPLN,
    TLS209_APPLN_IPC, TLS224_APPLN_CPC, TLS211_PAT_PUBLN
)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from sqlalchemy import func
from datetime import datetime
import re
import os
import warnings
warnings.filterwarnings('ignore')

# Connect to PATSTAT TIP database
patstat = PatstatClient(env='PROD')
db = patstat.orm()

print("✅ Libraries and PATSTAT connection initialized")
print(f"🕐 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries and PATSTAT connection initialized
🕐 Analysis started at: 2026-03-11 08:39:43


In [2]:
# ==============================================
# LOAD ELECTROLYZER DATASET
# ==============================================

# Load electrolyzer dataset to get docdb_family_id values
print("📊 Loading electrolyzer dataset...")
electrolyzer_df = pd.read_excel('03_Dataset_Enhancement__Elettrolizzatori_Enhanced_Final_Dataset_2025!!!.xlsx')
print(f"✅ Loaded {len(electrolyzer_df)} electrolyzer patents")

# Extract unique docdb_family_id values
electrolyzer_family_ids = electrolyzer_df['docdb_family_id'].unique().tolist()
print(f"📋 Found {len(electrolyzer_family_ids)} unique patent families")
print(f"🔍 Sample family IDs: {electrolyzer_family_ids[:10]}")

# Configuration parameters
START_YEAR = 2000
END_YEAR = 2023
BATCH_SIZE_YEARS = 3

print(f"\n⚙️ Configuration:")
print(f"   📅 Year range: {START_YEAR}-{END_YEAR}")
print(f"   🔢 Patent families to process: {len(electrolyzer_family_ids)}")
print(f"   📝 Mode: Enhanced extraction with sector information")

📊 Loading electrolyzer dataset...
✅ Loaded 18811 electrolyzer patents
📋 Found 18811 unique patent families
🔍 Sample family IDs: [3819525, 3824194, 3832658, 4168002, 4168863, 4169054, 4169160, 4170032, 4581634, 4583816]

⚙️ Configuration:
   📅 Year range: 2000-2023
   🔢 Patent families to process: 18811
   📝 Mode: Enhanced extraction with sector information


In [3]:
# ==============================================
# ENHANCED DATA EXTRACTION WITH SECTOR INFO
# ==============================================

def extract_batch_data_enhanced(start_year, end_year):
    """
    Extract enhanced patent applicant data with sector information for a specific year range
    """
    print(f"  🔍 Processing years {start_year}-{end_year}...")
    
    try:
        # Filter electrolyzer families that fall within the year range
        family_ids_in_range = (
            db.query(TLS201_APPLN.docdb_family_id)
            .filter(
                TLS201_APPLN.docdb_family_id.in_(electrolyzer_family_ids),
                TLS201_APPLN.earliest_filing_year.between(start_year, end_year)
            )
            .distinct()
        ).all()
        
        family_ids_list = [row[0] for row in family_ids_in_range]
        
        if not family_ids_list:
            print(f"    ⚠️ No electrolyzer patents found for {start_year}-{end_year}")
            return pd.DataFrame()
        
        print(f"    📋 Found {len(family_ids_list)} electrolyzer families in range")
        
        # Enhanced query with sector information from TLS206_PERSON
        query = db.query(
            TLS206_PERSON.psn_name,
            TLS206_PERSON.psn_sector,  # This is the key enhancement!
            TLS201_APPLN.docdb_family_id,
            TLS201_APPLN.earliest_filing_year,
            TLS207_PERS_APPLN.person_id
        ).join(
            TLS207_PERS_APPLN, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id
        ).join(
            TLS201_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id
        ).filter(
            TLS207_PERS_APPLN.applt_seq_nr != 0,
            TLS201_APPLN.docdb_family_id.in_(family_ids_list),
            TLS201_APPLN.earliest_filing_year.between(start_year, end_year)
        ).order_by(
            TLS206_PERSON.psn_name,
            TLS201_APPLN.docdb_family_id
        )
        
        result = query.all()
        
        if result:
            df = pd.DataFrame(result, columns=['Applicant_Name', 'PSN_Sector', 'Patent_Family_ID', 'Filing_Year', 'Person_ID'])
            print(f"    ✅ Found {len(df)} applicant-patent relationships with sector info")
            return df
        else:
            print(f"    ⚠️ No applicant data found for {start_year}-{end_year}")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"    ❌ Error processing {start_year}-{end_year}: {str(e)}")
        return pd.DataFrame()

# Extract data in batches and combine
print("🚀 Starting enhanced electrolyzer patent data extraction...")
all_batches = []

for year in range(START_YEAR, END_YEAR + 1, BATCH_SIZE_YEARS):
    batch_end = min(year + BATCH_SIZE_YEARS - 1, END_YEAR)
    df_batch = extract_batch_data_enhanced(year, batch_end)
    if not df_batch.empty:
        all_batches.append(df_batch)

if all_batches:
    print(f"\n📊 Combining {len(all_batches)} data batches...")
    raw_data = pd.concat(all_batches, ignore_index=True)
    
    print(f"✅ Enhanced data extraction complete!")
    print(f"   📈 Total applicant-patent relationships: {len(raw_data)}")
    print(f"   🏢 Unique applicant names: {raw_data['Applicant_Name'].nunique()}")
    print(f"   🏭 Unique patent families: {raw_data['Patent_Family_ID'].nunique()}")
    print(f"   📅 Year range in data: {raw_data['Filing_Year'].min()}-{raw_data['Filing_Year'].max()}")
    print(f"   🏷️ Applicants with sector info: {raw_data['PSN_Sector'].notna().sum()} / {len(raw_data)}")
    
else:
    print("❌ No data found! Please check the electrolyzer family IDs and year range.")
    raw_data = pd.DataFrame()

🚀 Starting enhanced electrolyzer patent data extraction...
  🔍 Processing years 2000-2002...
    📋 Found 695 electrolyzer families in range
    ✅ Found 1867 applicant-patent relationships with sector info
  🔍 Processing years 2003-2005...
    📋 Found 831 electrolyzer families in range
    ✅ Found 2073 applicant-patent relationships with sector info
  🔍 Processing years 2006-2008...
    📋 Found 1034 electrolyzer families in range
    ✅ Found 3296 applicant-patent relationships with sector info
  🔍 Processing years 2009-2011...
    📋 Found 1298 electrolyzer families in range
    ✅ Found 3868 applicant-patent relationships with sector info
  🔍 Processing years 2012-2014...
    📋 Found 1508 electrolyzer families in range
    ✅ Found 3222 applicant-patent relationships with sector info
  🔍 Processing years 2015-2017...
    📋 Found 2685 electrolyzer families in range
    ✅ Found 4603 applicant-patent relationships with sector info
  🔍 Processing years 2018-2020...
    📋 Found 3492 electrolyz

In [4]:
# ==============================================
# ENHANCED SECTOR CLASSIFICATION FUNCTION
# ==============================================

def classify_applicant_sector(name, psn_sector=None):
    """
    Classify applicant sector using database psn_sector first, then enhanced name-based fallback
    Priority: Database psn_sector classification > Enhanced name-based fallback
    Special handling: Use name-based classification for vague psn_sector values like 'OTHER' and 'UNKNOWN'
    """
    # First priority: Use database psn_sector if available and not vague
    if pd.notna(psn_sector) and psn_sector.strip():
        psn_upper = psn_sector.upper()
        
        # Skip vague database classifications and use name-based instead
        vague_classifications = ['OTHER', 'UNKNOWN', 'UNDEFINED', 'NOT SPECIFIED', 'N/A']
        if psn_upper not in vague_classifications:
            sector_map = {
                'COMPANY': 'Company',
                'INDIVIDUAL': 'Individual',
                'UNIVERSITY': 'University',
                'GOV NON-PROFIT': 'Government/Non-Profit',
                'GOVERNMENT': 'Government/Non-Profit',
                'HOSPITAL': 'Healthcare/Hospital',
                'RESEARCH INSTITUTION': 'Research Institution',
                'RESEARCH': 'Research Institution'
            }
            mapped_sector = sector_map.get(psn_upper)
            if mapped_sector is not None:
                return mapped_sector
        # If psn_sector is vague or unmapped, fall through to name-based classification
    
    # Enhanced name-based classification (used as fallback or for vague PSN_SECTOR values)
    if pd.isna(name) or not name.strip():
        return 'Unknown'
    
    name_upper = name.upper()
    
    # Academy of Sciences and similar research institutions (highest priority)
    academy_keywords = [
        'ACADEMY OF SCIENCES', 'CHINESE ACADEMY', 'ROYAL SOCIETY', 'NATIONAL ACADEMY',
        'ACADEMIA SINICA', 'KOREAN ACADEMY', 'RUSSIAN ACADEMY', 'KOREAN ACADEMY OF SCIENCE'
    ]
    
    # Research institution indicators (comprehensive)
    research_keywords = [
        'RESEARCH INSTITUTE', 'RESEARCH CENTER', 'RESEARCH CENTRE', 'LABORATORY',
        'NATIONAL INSTITUTE', 'FRAUNHOFER', 'CNRS', 'RIKEN', 'MAX PLANCK',
        'RESEARCH FOUNDATION', 'SCIENTIFIC RESEARCH', 'TECHNOLOGY RESEARCH',
        'CLEAN ENERGY RESEARCH', 'ENERGY RESEARCH', 'THERMAL POWER RESEARCH',
        'POWER RESEARCH INSTITUTE', 'RESEARCH AND DEVELOPMENT', 'R&D CENTER'
    ]
    
    # University indicators
    university_keywords = [
        'UNIVERSITY', 'UNIVERSITE', 'UNIVERSITEIT', 'UNIVERSIDAD', 'UNIVERSITAET',
        'COLLEGE', 'SCHOOL OF', 'TECHNICAL UNIVERSITY', 'POLYTECHNIC UNIVERSITY',
        'INSTITUTE OF TECHNOLOGY', 'TECHNICAL INSTITUTE'
    ]
    
    # Government/Public indicators
    government_keywords = [
        'MINISTRY', 'GOVERNMENT', 'STATE GRID', 'NATIONAL', 'STATE', 'FEDERAL', 'PUBLIC',
        'COMMISSARIAT', 'AGENCY', 'ADMINISTRATION', 'DEPARTMENT', 'MUNICIPAL',
        'PROVINCIAL', 'REGIONAL'
    ]
    
    # Healthcare indicators
    healthcare_keywords = ['HOSPITAL', 'MEDICAL', 'HEALTH', 'CLINIC']
    
    # Individual indicators (enhanced)
    individual_indicators = [',', 'DR.', 'PROF.', 'MR.', 'MS.', 'MRS.']
    if (len(name.split()) <= 3 and 
        any(indicator in name_upper for indicator in individual_indicators)):
        return 'Individual'
    
    # Check categories in priority order
    # 1. Academy of Sciences (highest priority for research classification)
    if any(keyword in name_upper for keyword in academy_keywords):
        return 'Research Institution'
    
    # 2. Research institutions (before universities to catch research-focused institutes)
    elif any(keyword in name_upper for keyword in research_keywords):
        return 'Research Institution'
    
    # 3. Universities
    elif any(keyword in name_upper for keyword in university_keywords):
        return 'University'
    
    # 4. Government/Public entities
    elif any(keyword in name_upper for keyword in government_keywords):
        return 'Government/Non-Profit'
    
    # 5. Healthcare
    elif any(keyword in name_upper for keyword in healthcare_keywords):
        return 'Healthcare/Hospital'
    
    # 6. Default to Company
    else:
        return 'Company'

print("✅ Enhanced sector classification function defined successfully!")
print("🔍 Features:")
print("   • Prioritizes database PSN_SECTOR when reliable")
print("   • Falls back to name-based classification for vague values ('OTHER', 'UNKNOWN')")
print("   • Enhanced detection of Research Institutions and Academies of Sciences")
print("   • Improved keyword matching for all sectors")

✅ Enhanced sector classification function defined successfully!
🔍 Features:
   • Prioritizes database PSN_SECTOR when reliable
   • Falls back to name-based classification for vague values ('OTHER', 'UNKNOWN')
   • Enhanced detection of Research Institutions and Academies of Sciences
   • Improved keyword matching for all sectors


In [5]:
# ==============================================
# APPLY SECTOR CLASSIFICATION
# ==============================================

if not raw_data.empty:
    print("🏷️ Applying enhanced sector classification...")
    
    # Apply sector classification using the enhanced function
    raw_data['Sector'] = raw_data.apply(
        lambda row: classify_applicant_sector(row['Applicant_Name'], row['PSN_Sector']), 
        axis=1
    )
    
    # Count classification sources
    db_classified = raw_data['PSN_Sector'].notna().sum()
    name_classified = raw_data['PSN_Sector'].isna().sum()
    vague_psn = raw_data[raw_data['PSN_Sector'].isin(['OTHER', 'UNKNOWN', 'UNDEFINED'])]['PSN_Sector'].count()
    
    print(f"✅ Enhanced sector classification complete!")
    print(f"   🗄️ Database sector info: {db_classified} records")
    print(f"   📝 Name-based classification: {name_classified} records")
    print(f"   🔄 Vague PSN_SECTOR values (using name-based): {vague_psn} records")
    
    # Show sector distribution
    sector_counts = raw_data['Sector'].value_counts()
    print(f"\n📊 Sector distribution:")
    for sector, count in sector_counts.items():
        percentage = (count / len(raw_data)) * 100
        print(f"   {sector}: {count:,} ({percentage:.1f}%)")
        
    # Show examples of improved classification
    print(f"\n🔍 Examples of sector classification:")
    examples = [
        'CHINESE ACADEMY OF SCIENCES',
        'HUANENG CLEAN ENERGY RESEARCH INSTITUTE', 
        'TSINGHUA UNIVERSITY',
        'HONDA MOTOR COMPANY'
    ]
    
    for example in examples:
        matches = raw_data[raw_data['Applicant_Name'] == example]
        if not matches.empty:
            sector = matches.iloc[0]['Sector']
            psn_sector = matches.iloc[0]['PSN_Sector']
            print(f"   📌 {example[:50]}... → {sector} (DB: {psn_sector})")
else:
    print("⚠️ No data available for sector classification.")

🏷️ Applying enhanced sector classification...
✅ Enhanced sector classification complete!
   🗄️ Database sector info: 36806 records
   📝 Name-based classification: 0 records
   🔄 Vague PSN_SECTOR values (using name-based): 5931 records

📊 Sector distribution:
   Company: 22,181 (60.3%)
   Individual: 8,374 (22.8%)
   University: 4,198 (11.4%)
   Government/Non-Profit: 1,182 (3.2%)
   Research Institution: 769 (2.1%)
   Healthcare/Hospital: 102 (0.3%)

🔍 Examples of sector classification:
   📌 CHINESE ACADEMY OF SCIENCES... → Research Institution (DB: GOV NON-PROFIT UNIVERSITY)
   📌 HUANENG CLEAN ENERGY RESEARCH INSTITUTE... → Company (DB: COMPANY)
   📌 TSINGHUA UNIVERSITY... → University (DB: UNIVERSITY)
   📌 HONDA MOTOR COMPANY... → Company (DB: COMPANY)


In [6]:
# ==============================================
# AGGREGATED APPLICANT SUMMARY WITH SECTORS
# ==============================================

if not raw_data.empty:
    print("📊 Creating enhanced applicant summary with sector information...")
    
    # Aggregate data by applicant name
    applicant_summary = raw_data.groupby(['Applicant_Name', 'Sector']).agg({
        'Patent_Family_ID': 'nunique',  # Count unique patent families
        'Filing_Year': ['min', 'max'],   # Get year range
        'Person_ID': 'nunique'          # Count unique person IDs
    }).reset_index()
    
    # Flatten column names
    applicant_summary.columns = ['Applicant_Name', 'Sector', 'Patent_Families', 'First_Year', 'Last_Year', 'Person_IDs']
    
    # Sort by number of patent families (descending)
    applicant_summary = applicant_summary.sort_values('Patent_Families', ascending=False).reset_index(drop=True)
    
    # Add rank
    applicant_summary['Rank'] = range(1, len(applicant_summary) + 1)

    # Build per-applicant list of unique patent family IDs.
    # Embedded in the HTML so the browser can compute a deduplicated total
    # and detect co-applicant overlaps when multiple applicants are selected.
    fam_lists = (
        raw_data.groupby(['Applicant_Name', 'Sector'])['Patent_Family_ID']
        .apply(lambda x: sorted(int(v) for v in x.unique()))
        .reset_index()
        .rename(columns={'Patent_Family_ID': 'Family_ID_List'})
    )
    applicant_summary = applicant_summary.merge(
        fam_lists, on=['Applicant_Name', 'Sector'], how='left'
    )
    applicant_summary['Family_ID_List'] = applicant_summary['Family_ID_List'].apply(
        lambda x: x if isinstance(x, list) else []
    )
    
    print(f"✅ Enhanced summary created!")
    print(f"   🏢 Total unique applicants: {len(applicant_summary)}")
    print(f"   📊 Patent families range: {applicant_summary['Patent_Families'].min()}-{applicant_summary['Patent_Families'].max()}")
    print(f"   🏷️ Sectors identified: {applicant_summary['Sector'].nunique()}")
    

    # ── WARNING: applicants split across multiple sector labels ──────────────
    # Because applicant_summary is grouped by (Applicant_Name, Sector), the same
    # real-world entity can appear more than once if PATSTAT assigns different
    # psn_sector values across its patent records.  When this happens the family
    # count shown on each card in the HTML reflects only one (Name, Sector) row,
    # while the co-applicant overlap table merges all rows by name and may report
    # a slightly higher total.  This faithfully mirrors the original PATSTAT
    # classification and is NOT corrected automatically, but users should verify
    # the sector labels for any flagged applicant before drawing conclusions.
    multi_sector = (
        applicant_summary.groupby('Applicant_Name')['Sector']
        .nunique()
        .reset_index()
        .rename(columns={'Sector': 'n_sectors'})
    )
    multi_sector = multi_sector[multi_sector['n_sectors'] > 1].sort_values('n_sectors', ascending=False)
    if not multi_sector.empty:
        print(f"\n⚠️  DATA QUALITY WARNING – {len(multi_sector)} applicant(s) appear under MORE THAN ONE sector label in PATSTAT.")
        print(   "   This causes a minor discrepancy between the per-card family count in the HTML")
        print(   "   and the 'Total' column in the co-applicant overlap table (which merges all rows).")
        print(   "   Please review these applicants and verify their sector classification:")
        for _, row in multi_sector.iterrows():
            name = row['Applicant_Name']
            sectors = applicant_summary.loc[
                applicant_summary['Applicant_Name'] == name, ['Sector', 'Patent_Families']
            ].to_dict('records')
            sector_str = ', '.join(f"{s['Sector']} ({s['Patent_Families']} fam.)" for s in sectors)
            print(f"   • {name}: {sector_str}")
    else:
        print("\n✅ No applicants with conflicting sector labels found.")
    # ─────────────────────────────────────────────────────────────────────────
    # Show final sector distribution from summary
    final_sector_counts = applicant_summary.groupby('Sector')['Patent_Families'].sum().sort_values(ascending=False)
    total_families = final_sector_counts.sum()
    print(f"\n📊 Final Sector Distribution in Summary:")
    for sector, count in final_sector_counts.items():
        percentage = (count / total_families) * 100
        print(f"   {sector}: {count:,} families ({percentage:.1f}%)")
    
else:
    print("⚠️ No data available for analysis.")
    applicant_summary = pd.DataFrame()

📊 Creating enhanced applicant summary with sector information...
✅ Enhanced summary created!
   🏢 Total unique applicants: 12849
   📊 Patent families range: 1-308
   🏷️ Sectors identified: 6

⚠️  DATA QUALITY WARNING – 49 applicant(s) appear under MORE THAN ONE sector label in PATSTAT.
   This causes a minor discrepancy between the per-card family count in the HTML
   and the 'Total' column in the co-applicant overlap table (which merges all rows).
   Please review these applicants and verify their sector classification:
   • ALLIANCE MAGNESIUM: Company (2 fam.), Individual (2 fam.)
   • AUGE II WAYNE K: Individual (3 fam.), Company (1 fam.)
   • AVRIL: Individual (1 fam.), Company (1 fam.)
   • DALIAN SHUANGDI CREATIVE TECHNOLOGY RESEARCH INSTITUTE COMPANY: Company (10 fam.), Government/Non-Profit (1 fam.)
   • DALIAN SHUANGDI INNOVATIVE TECHNOLOGY RESEARCH INSTITUTE COMPANY: Government/Non-Profit (9 fam.), Company (8 fam.)
   • FAIRLIE MATTHEW: Company (1 fam.), Individual (1 fam.)
 

# Siemens Name Harmonisation

Applying the Eurostat/ECOOM name harmonisation methodology (KS-RA-11-008-EN) to identify and consolidate all Siemens name variants in the electrolyzer patent dataset.

**Methodology Layers (following Eurostat):**
1. Character Cleaning (HTML codes, special chars to ASCII)
2. Punctuation Cleaning (normalise separators)
3. Legal Form Removal (AG, GmbH, Co. KG, Inc., Ltd., S.A., etc.)
4. Common Company Word Removal (Company, Corporation, Gesellschaft, etc.)
5. Spelling Variation Harmonisation (System/Systems/Systeme/Systemen)
6. Condensing (remove non-alphanumeric chars)
7. Umlaut Harmonisation
8. Matching & Group Assignment

In [7]:
# ==============================================
# EUROSTAT-STYLE NAME HARMONISATION PIPELINE
# ==============================================

import unicodedata

def harmonise_name(name):
    """
    Apply Eurostat/ECOOM name harmonisation steps to a patentee name.
    Returns dict with intermediate results for each step.
    """
    steps = {}
    steps['original'] = name

    # --- Step 1: Character Cleaning ---
    s = name
    s = s.replace('&AMP;', '&').replace('&amp;', '&')
    s = s.replace('<BR>', ' ').replace('<br>', ' ').replace('\n', ' ').replace('\r', ' ')
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = s.upper().strip()
    steps['char_cleaned'] = s

    # --- Step 2: Punctuation Cleaning ---
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'\s*,\s*', ', ', s)
    s = re.sub(r'\.\s*', '. ', s).strip()
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.rstrip('.,;:')
    steps['punct_cleaned'] = s

    # --- Step 3: Legal Form Removal ---
    legal_forms = [
        r'\bAKTIENGESELLSCHAFT\b', r'\bA\.?\s*G\.?\b',
        r'\bGMBH\s*&\s*CO\.?\s*,?\s*K\.?\s*G\.?\b', r'\bGMBH\b',
        r'\bCO\.?\s*,?\s*K\.?\s*G\.?\b',
        r'\bCORPORATION\b', r'\bCORP\.?\b', r'\bINCORPORATED\b', r'\bINC\.?\b',
        r'\bLIMITED\b', r'\bLTD\.?\b', r'\bL\.?\s*L\.?\s*C\.?\b',
        r'\bP\.?\s*L\.?\s*C\.?\b', r'\bCO\.?\b',
        r'\bS\.?\s*A\.?\s*R\.?\s*L\.?\b', r'\bS\.?\s*A\.?\s*S\.?\b',
        r'\bS\.?\s*A\.?\b',
        r'\bS\.?\s*P\.?\s*A\.?\b', r'\bS\.?\s*R\.?\s*L\.?\b',
        r'\bB\.?\s*V\.?\b', r'\bN\.?\s*V\.?\b',
        r'\bA\.?\s*S\.?\b', r'\bA/S\b',
    ]
    s_no_legal = s
    for pattern in legal_forms:
        s_no_legal = re.sub(pattern, '', s_no_legal)
    s_no_legal = re.sub(r'\s*&\s*$', '', s_no_legal)
    s_no_legal = re.sub(r'^\s*&\s*', '', s_no_legal)
    s_no_legal = re.sub(r'\s+', ' ', s_no_legal).strip()
    s_no_legal = s_no_legal.rstrip('.,;:& ')
    steps['legal_removed'] = s_no_legal

    # --- Step 4: Common Company Word Removal ---
    common_words = [
        r'\bCOMPANY\b', r'\bGESELLSCHAFT\b', r'\bSOCIETE\b', r'\bSOCIEDAD\b',
        r'\bENTERPRISE[S]?\b', r'\bHOLDING[S]?\b', r'\bGROUP\b',
        r'\bINTERNATIONAL\b', r'\bGLOBAL\b',
    ]
    s_no_common = s_no_legal
    for pattern in common_words:
        s_no_common = re.sub(pattern, '', s_no_common)
    s_no_common = re.sub(r'\s+', ' ', s_no_common).strip()
    s_no_common = s_no_common.rstrip('.,;:& ')
    steps['common_removed'] = s_no_common

    # --- Step 5: Spelling Variation Harmonisation ---
    spelling_map = {
        r'\bSYSTEMS?\b': 'SYSTEM', r'\bSYSTEME[NS]?\b': 'SYSTEM',
        r'\bTECHNOLOGIES\b': 'TECHNOLOGY', r'\bTECHNOLOGIEN\b': 'TECHNOLOGY',
        r'\bENERGIE[S]?\b': 'ENERGY', r'\bRENEWABLE[S]?\b': 'RENEWABLE',
        r'\bINDUSTRIES\b': 'INDUSTRY',
    }
    s_spell = s_no_common
    for pattern, repl in spelling_map.items():
        s_spell = re.sub(pattern, repl, s_spell)
    s_spell = re.sub(r'\s+', ' ', s_spell).strip()
    steps['spelling_harmonised'] = s_spell

    # --- Step 6: Condensing ---
    s_condensed = re.sub(r'[^A-Z0-9]', '', s_spell)
    steps['condensed'] = s_condensed

    # --- Step 7: Umlaut Harmonisation ---
    s_umlaut = s_condensed
    s_umlaut = s_umlaut.replace('UE', 'U').replace('AE', 'A').replace('OE', 'O')
    steps['umlaut_harmonised'] = s_umlaut

    return steps

# Test
test = harmonise_name("Siemens Energy Global GmbH & Co. KG")
print("Example harmonisation steps:")
for step, val in test.items():
    print(f"  {step:25s} -> {val}")


Example harmonisation steps:
  original                  -> Siemens Energy Global GmbH & Co. KG
  char_cleaned              -> SIEMENS ENERGY GLOBAL GMBH & CO. KG
  punct_cleaned             -> SIEMENS ENERGY GLOBAL GMBH & CO. KG
  legal_removed             -> SIEMENS ENERGY GLOBAL
  common_removed            -> SIEMENS ENERGY
  spelling_harmonised       -> SIEMENS ENERGY
  condensed                 -> SIEMENSENERGY
  umlaut_harmonised         -> SIEMENSENERGY


In [8]:
# ==============================================
# FIND & HARMONISE ALL SIEMENS NAME VARIANTS
# ==============================================

siemens_pattern = r'SIEMENS|VOITH\s+SIEMENS|BSH.*BOSCH.*SIEMENS'
siemens_mask = raw_data['Applicant_Name'].str.upper().str.contains(siemens_pattern, regex=True, na=False)
siemens_raw = raw_data[siemens_mask].copy()

print(f"Found {len(siemens_raw):,} raw records matching Siemens pattern")
print(f"   Unique name variants: {siemens_raw['Applicant_Name'].nunique()}")
print(f"   Unique patent families: {siemens_raw['Patent_Family_ID'].nunique()}")

# Apply harmonisation to each unique name
unique_names = siemens_raw['Applicant_Name'].unique()
harmonisation_results = []

for name in unique_names:
    steps = harmonise_name(name)
    fam_count = siemens_raw[siemens_raw['Applicant_Name'] == name]['Patent_Family_ID'].nunique()
    steps['patent_families'] = fam_count
    harmonisation_results.append(steps)

harm_df = pd.DataFrame(harmonisation_results)

print(f"\nHarmonisation Pipeline Results ({len(harm_df)} name variants):")
print("=" * 100)
for _, row in harm_df.sort_values('patent_families', ascending=False).iterrows():
    print(f"\n  Original:    {row['original']}")
    if row['legal_removed'] != row['char_cleaned']:
        print(f"  Legal form:  {row['legal_removed']}")
    if row['common_removed'] != row['legal_removed']:
        print(f"  Common word: {row['common_removed']}")
    if row['spelling_harmonised'] != row['common_removed']:
        print(f"  Spelling:    {row['spelling_harmonised']}")
    print(f"  Condensed:   {row['condensed']}")
    if row['umlaut_harmonised'] != row['condensed']:
        print(f"  Umlaut:      {row['umlaut_harmonised']}")
    print(f"  Families:    {row['patent_families']}")


Found 442 raw records matching Siemens pattern
   Unique name variants: 18
   Unique patent families: 97

Harmonisation Pipeline Results (18 name variants):

  Original:    SIEMENS ENERGY GLOBAL & COMPANY
  Common word: SIEMENS ENERGY
  Condensed:   SIEMENSENERGY
  Families:    44

  Original:    SIEMENS
  Condensed:   SIEMENS
  Families:    42

  Original:    SIEMENS GAMESA RENEWABLE ENERGY
  Condensed:   SIEMENSGAMESARENEWABLEENERGY
  Families:    15

  Original:    SIEMENS ENERGY INTERNATIONAL, INC.
  Legal form:  SIEMENS ENERGY INTERNATIONAL
  Common word: SIEMENS ENERGY
  Condensed:   SIEMENSENERGY
  Families:    11

  Original:    Siemens Energy Global GmbH & Co. KG
  Legal form:  SIEMENS ENERGY GLOBAL
  Common word: SIEMENS ENERGY
  Condensed:   SIEMENSENERGY
  Families:    8

  Original:    SIEMENS ENERGY GLOBAL GMBH & CO. KG
  Legal form:  SIEMENS ENERGY GLOBAL
  Common word: SIEMENS ENERGY
  Condensed:   SIEMENSENERGY
  Families:    5

  Original:    SIEMENS GAMESA RENEWABLE 

In [9]:
# ==============================================
# GROUP ASSIGNMENT VIA CONDENSED NAMES
# ==============================================

def assign_siemens_group(condensed_name, original_name):
    c = condensed_name
    o = original_name.upper()
    
    # Joint ventures first (most specific)
    if 'VOITH' in o and 'SIEMENS' in o:
        return 'Voith Siemens (JV)'
    if 'BSH' in o or ('BOSCH' in o and 'SIEMENS' in o):
        return 'BSH (Bosch/Siemens JV)'
    if 'SIEMENSVAI' in c or 'VAI' in o:
        return 'Siemens VAI (JV)'
    if 'GAMESA' in c:
        return 'Siemens Gamesa'
    if 'ENERGY' in c or 'ENERGY' in o:
        return 'Siemens Energy'
    return 'Siemens AG'

harm_df['group'] = harm_df.apply(
    lambda r: assign_siemens_group(r['condensed'], r['original']), axis=1
)

name_to_group = dict(zip(harm_df['original'], harm_df['group']))
name_to_condensed = dict(zip(harm_df['original'], harm_df['condensed']))
siemens_raw['Siemens_Group'] = siemens_raw['Applicant_Name'].map(name_to_group)
siemens_raw['Harmonised_Name'] = siemens_raw['Applicant_Name'].map(name_to_condensed)

print("Corporate Group Assignment:")
print("=" * 90)
for group in ['Siemens AG', 'Siemens Energy', 'Siemens Gamesa', 'Voith Siemens (JV)', 'BSH (Bosch/Siemens JV)', 'Siemens VAI (JV)']:
    group_df = harm_df[harm_df['group'] == group].sort_values('patent_families', ascending=False)
    if group_df.empty:
        continue
    total_fam = group_df['patent_families'].sum()
    n_variants = len(group_df)
    print(f"\n  {group}  ({n_variants} name variants, {total_fam} patent families)")
    for _, row in group_df.iterrows():
        print(f"   {row['original']:60s} -> {row['condensed']:30s} ({row['patent_families']} fam)")

print(f"\n\nHarmonisation Impact:")
print(f"   Original unique names:     {len(harm_df)}")
print(f"   After condensing:          {harm_df['condensed'].nunique()}")
print(f"   Corporate groups:          {harm_df['group'].nunique()}")
reduction = (1 - harm_df['condensed'].nunique() / len(harm_df)) * 100
print(f"   Name reduction:            {reduction:.1f}%")


Corporate Group Assignment:

  Siemens AG  (2 name variants, 43 patent families)
   SIEMENS                                                      -> SIEMENS                        (42 fam)
   Siemens Aktiengesellschaft                                   -> SIEMENS                        (1 fam)

  Siemens Energy  (7 name variants, 71 patent families)
   SIEMENS ENERGY GLOBAL & COMPANY                              -> SIEMENSENERGY                  (44 fam)
   SIEMENS ENERGY INTERNATIONAL, INC.                           -> SIEMENSENERGY                  (11 fam)
   Siemens Energy Global GmbH & Co. KG                          -> SIEMENSENERGY                  (8 fam)
   SIEMENS ENERGY GLOBAL GMBH & CO. KG                          -> SIEMENSENERGY                  (5 fam)
   Siemens Energy Global GmbH & Co. \n KG                       -> SIEMENSENERGYNKG               (1 fam)
   SIEMENS ENERGY                                               -> SIEMENSENERGY                  (1 fam)
   SIEMENS 

In [10]:
# ==============================================
# COMPETITIVE CONTEXT
# ==============================================

siemens_summary_mask = applicant_summary['Applicant_Name'].str.upper().str.contains(
    r'SIEMENS|VOITH\s+SIEMENS|BSH.*BOSCH.*SIEMENS', regex=True, na=False
)
siemens_summary = applicant_summary[siemens_summary_mask].copy()
siemens_summary['Siemens_Group'] = siemens_summary['Applicant_Name'].map(name_to_group)

siemens_total = siemens_raw['Patent_Family_ID'].nunique()
total_families_all = applicant_summary['Patent_Families'].sum()
siemens_rank = (applicant_summary['Patent_Families'] >= siemens_total).sum()

print(f"Siemens (all entities combined): {siemens_total} unique patent families")
print(f"   Share of total electrolyzer patents: {siemens_total/total_families_all*100:.1f}%")
print(f"   Approximate rank (if combined): #{siemens_rank}")

print(f"\nBy Corporate Group:")
group_agg = siemens_raw.groupby('Siemens_Group').agg(
    families=('Patent_Family_ID', 'nunique'),
    names=('Applicant_Name', 'nunique'),
    first_year=('Filing_Year', 'min'),
    last_year=('Filing_Year', 'max'),
).sort_values('families', ascending=False)

for group, row in group_agg.iterrows():
    print(f"   {group:30s}  {row['families']:4d} families  ({row['names']} names, {row['first_year']}-{row['last_year']})")


Siemens (all entities combined): 97 unique patent families
   Share of total electrolyzer patents: 0.4%
   Approximate rank (if combined): #13

By Corporate Group:
   Siemens Energy                    46 families  (7 names, 2015-2023)
   Siemens AG                        42 families  (2 names, 2000-2023)
   Siemens Gamesa                    15 families  (4 names, 2019-2023)
   BSH (Bosch/Siemens JV)             1 families  (1 names, 2012-2012)
   Siemens VAI (JV)                   1 families  (1 names, 2004-2004)
   Voith Siemens (JV)                 1 families  (3 names, 2002-2002)


In [11]:
# ==============================================
# TECHNOLOGY PROFILE & GEOGRAPHIC STRATEGY
# ==============================================

siemens_family_ids = siemens_raw['Patent_Family_ID'].unique().tolist()

print("Querying IPC classifications...")
ipc_query = db.query(
    TLS209_APPLN_IPC.ipc_class_symbol,
    TLS201_APPLN.docdb_family_id,
    TLS201_APPLN.earliest_filing_year
).join(
    TLS201_APPLN, TLS209_APPLN_IPC.appln_id == TLS201_APPLN.appln_id
).filter(
    TLS201_APPLN.docdb_family_id.in_(siemens_family_ids)
).all()

ipc_df = pd.DataFrame(ipc_query, columns=['IPC_Symbol', 'Family_ID', 'Filing_Year'])
ipc_df['IPC_Class'] = ipc_df['IPC_Symbol'].str.strip().str[:4]
print(f"Retrieved {len(ipc_df)} IPC classifications for {ipc_df['Family_ID'].nunique()} families")

ipc_top = ipc_df.groupby('IPC_Class')['Family_ID'].nunique().sort_values(ascending=False).head(15)
print(f"\nTop 15 IPC Classes:")
for ipc, count in ipc_top.items():
    print(f"   {ipc}: {count} families")

try:
    cpc_query = db.query(
        TLS224_APPLN_CPC.cpc_class_symbol,
        TLS201_APPLN.docdb_family_id,
    ).join(
        TLS201_APPLN, TLS224_APPLN_CPC.appln_id == TLS201_APPLN.appln_id
    ).filter(
        TLS201_APPLN.docdb_family_id.in_(siemens_family_ids)
    ).all()
    cpc_df = pd.DataFrame(cpc_query, columns=['CPC_Symbol', 'Family_ID'])
    cpc_df['CPC_Class'] = cpc_df['CPC_Symbol'].str.strip().str[:4]
    print(f"\nTop 15 CPC Classes:")
    cpc_top = cpc_df.groupby('CPC_Class')['Family_ID'].nunique().sort_values(ascending=False).head(15)
    for cpc, count in cpc_top.items():
        print(f"   {cpc}: {count} families")
except Exception as e:
    print(f"CPC query failed: {e}")
    cpc_df = pd.DataFrame()

print("\nQuerying publication offices...")
publn_query = db.query(
    TLS211_PAT_PUBLN.publn_auth,
    TLS201_APPLN.docdb_family_id,
    TLS201_APPLN.earliest_filing_year
).join(
    TLS201_APPLN, TLS211_PAT_PUBLN.appln_id == TLS201_APPLN.appln_id
).filter(
    TLS201_APPLN.docdb_family_id.in_(siemens_family_ids)
).all()

publn_df = pd.DataFrame(publn_query, columns=['Publn_Auth', 'Family_ID', 'Filing_Year'])
office_counts = publn_df.groupby('Publn_Auth')['Family_ID'].nunique().sort_values(ascending=False)
print(f"\nPatent Office Distribution (Top 15):")
for office, count in office_counts.head(15).items():
    print(f"   {office}: {count} families")


Querying IPC classifications...
Retrieved 1683 IPC classifications for 97 families

Top 15 IPC Classes:
   C25B: 77 families
   F03D: 17 families
   H02J: 14 families
   C01B: 14 families
   C07C: 11 families
   F01K: 7 families
   B01J: 7 families
   B01D: 7 families
   H01M: 6 families
   C10G: 6 families
   C01C: 5 families
   H02K: 4 families
   C02F: 4 families
   C10J: 3 families
   H02P: 3 families

Top 15 CPC Classes:
   C25B: 75 families
   Y02E: 72 families
   Y02P: 28 families
   F05B: 16 families
   F03D: 16 families
   H02J: 12 families
   C07C: 9 families
   C01B: 8 families
   F01K: 7 families
   C10G: 6 families
   B01D: 6 families
   C01C: 5 families
   B01J: 4 families
   H01M: 4 families
   F22B: 4 families

Querying publication offices...

Patent Office Distribution (Top 15):
   EP: 77 families
   WO: 77 families
   CN: 59 families
   US: 53 families
   DE: 41 families
   AU: 26 families
   CA: 18 families
   DK: 15 families
   ES: 11 families
   JP: 10 families
   

In [12]:
# ==============================================
# VISUALISATIONS
# ==============================================

SIEMENS_GROUP_COLORS = {
    'Siemens AG': '#009999',
    'Siemens Energy': '#1f77b4',
    'Siemens Gamesa': '#2ca02c',
    'Voith Siemens (JV)': '#ff7f0e',
    'BSH (Bosch/Siemens JV)': '#d62728',
    'Siemens VAI (JV)': '#9467bd',
}

print("Creating visualisations...")

# 1. Sankey: Original names -> Corporate Groups
top_harm = harm_df[harm_df['patent_families'] >= 1].sort_values('patent_families', ascending=False)
groups = harm_df['group'].unique().tolist()

sankey_labels = top_harm['original'].tolist() + groups
name_idx = {n: i for i, n in enumerate(top_harm['original'])}
group_idx = {g: i + len(top_harm) for i, g in enumerate(groups)}

sankey_source, sankey_target, sankey_value = [], [], []
for _, row in top_harm.iterrows():
    sankey_source.append(name_idx[row['original']])
    sankey_target.append(group_idx[row['group']])
    sankey_value.append(row['patent_families'])

fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, thickness=20,
        label=sankey_labels,
        color=['#ccc'] * len(top_harm) + [SIEMENS_GROUP_COLORS.get(g, '#999') for g in groups]
    ),
    link=dict(
        source=sankey_source, target=sankey_target, value=sankey_value,
        color=['rgba({},{},{},0.5)'.format(int(SIEMENS_GROUP_COLORS.get(top_harm.iloc[i]['group'], '#999999')[1:3],16), int(SIEMENS_GROUP_COLORS.get(top_harm.iloc[i]['group'], '#999999')[3:5],16), int(SIEMENS_GROUP_COLORS.get(top_harm.iloc[i]['group'], '#999999')[5:7],16)) for i in range(len(top_harm))]
    )
)])
fig_sankey.update_layout(
    title='Name Harmonisation: Original Names -> Corporate Groups',
    height=max(400, len(top_harm) * 25),
    font=dict(size=11)
)
print("Sankey done")

# 2. Timeline by group
siemens_timeline = (
    siemens_raw.groupby(['Filing_Year', 'Siemens_Group'])['Patent_Family_ID']
    .nunique().reset_index().rename(columns={'Patent_Family_ID': 'Families'})
)

fig_timeline = go.Figure()
for group in ['Siemens AG', 'Siemens Energy', 'Siemens Gamesa', 'Voith Siemens (JV)', 'BSH (Bosch/Siemens JV)', 'Siemens VAI (JV)']:
    gd = siemens_timeline[siemens_timeline['Siemens_Group'] == group]
    if not gd.empty:
        fig_timeline.add_trace(go.Bar(
            x=gd['Filing_Year'], y=gd['Families'], name=group,
            marker_color=SIEMENS_GROUP_COLORS.get(group, '#999'),
        ))
fig_timeline.update_layout(
    title='Siemens Electrolyzer Patents by Year and Corporate Group',
    xaxis_title='Filing Year', yaxis_title='Patent Families',
    barmode='stack', height=500,
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig_timeline.update_xaxes(gridcolor='lightgray')
fig_timeline.update_yaxes(gridcolor='lightgray')
print("Timeline done")

# 3. IPC top 10
ipc_top10 = ipc_df.groupby('IPC_Class')['Family_ID'].nunique().sort_values(ascending=False).head(10)
fig_ipc = go.Figure()
fig_ipc.add_trace(go.Bar(
    x=ipc_top10.values, y=ipc_top10.index, orientation='h',
    marker_color='#009999', text=ipc_top10.values, textposition='auto'
))
fig_ipc.update_layout(
    title='Siemens - Top 10 IPC Classes',
    xaxis_title='Patent Families', yaxis_title='IPC Class',
    height=400, plot_bgcolor='white', paper_bgcolor='white',
    yaxis=dict(autorange='reversed'), margin=dict(l=100)
)
fig_ipc.update_xaxes(gridcolor='lightgray')
print("IPC done")

# 4. Geographic
office_top10 = publn_df.groupby('Publn_Auth')['Family_ID'].nunique().sort_values(ascending=False).head(10)
fig_geo = go.Figure()
fig_geo.add_trace(go.Bar(
    x=office_top10.index, y=office_top10.values,
    marker_color='#1f77b4', text=office_top10.values, textposition='auto'
))
fig_geo.update_layout(
    title='Siemens - Filing Offices (Top 10)',
    xaxis_title='Patent Office', yaxis_title='Patent Families',
    height=400, plot_bgcolor='white', paper_bgcolor='white',
)
fig_geo.update_yaxes(gridcolor='lightgray')
print("Geo done")

fig_sankey.show()
fig_timeline.show()
fig_ipc.show()
fig_geo.show()


Creating visualisations...
Sankey done
Timeline done
IPC done
Geo done


In [13]:
# ==============================================
# SIEMENS HTML REPORT
# ==============================================

html_filename = 'Siemens_Electrolyzer_Deep_Dive.html'

chart_divs = {
    'sankey': fig_sankey.to_html(include_plotlyjs=False, full_html=False),
    'timeline': fig_timeline.to_html(include_plotlyjs=False, full_html=False),
    'ipc': fig_ipc.to_html(include_plotlyjs=False, full_html=False),
    'geo': fig_geo.to_html(include_plotlyjs=False, full_html=False),
}

# Harmonisation detail table rows
harm_table_rows = ''
for _, row in harm_df.sort_values(['group', 'patent_families'], ascending=[True, False]).iterrows():
    color = SIEMENS_GROUP_COLORS.get(row['group'], '#999')
    harm_table_rows += (
        '<tr>'
        f'<td>{row["original"]}</td>'
        f'<td><code>{row["legal_removed"]}</code></td>'
        f'<td><code>{row["condensed"]}</code></td>'
        f'<td><span style="background:{color};color:white;padding:2px 8px;border-radius:10px;font-size:0.85em">{row["group"]}</span></td>'
        f'<td style="text-align:right">{row["patent_families"]}</td>'
        '</tr>\n'
    )

# IPC table
ipc_top15 = ipc_df.groupby('IPC_Class')['Family_ID'].nunique().sort_values(ascending=False).head(15)
ipc_table_rows = ''
for ipc, count in ipc_top15.items():
    ipc_table_rows += f'<tr><td>{ipc}</td><td style="text-align:right">{count}</td></tr>\n'

siemens_total = siemens_raw['Patent_Family_ID'].nunique()

html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Siemens Electrolyzer Patent Deep Dive</title>
    <script>__PLOTLY_JS_PLACEHOLDER__</script>
    <style>
        body {{ font-family: "Segoe UI", Tahoma, Geneva, Verdana, sans-serif; margin: 0; background: #f5f5f5; color: #333; }}
        .header {{ background: linear-gradient(135deg, #009999, #006666); color: white; padding: 50px 20px; text-align: center; }}
        .header h1 {{ margin: 0 0 10px 0; font-size: 2.2em; }}
        .header p {{ margin: 5px 0; opacity: 0.9; }}
        .container {{ max-width: 1200px; margin: 0 auto; padding: 20px; }}
        .section {{ background: white; border-radius: 10px; padding: 30px; margin: 20px 0; box-shadow: 0 2px 8px rgba(0,0,0,0.1); }}
        .section h2 {{ color: #006666; border-bottom: 2px solid #009999; padding-bottom: 10px; }}
        .chart-container {{ margin: 20px 0; }}
        table {{ border-collapse: collapse; width: 100%; font-size: 0.88em; }}
        th, td {{ border: 1px solid #ddd; padding: 6px 10px; text-align: left; }}
        th {{ background: #009999; color: white; position: sticky; top: 0; }}
        tr:nth-child(even) {{ background: #f9f9f9; }}
        tr:hover {{ background: #e0f5f5; }}
        code {{ background: #f0f0f0; padding: 1px 4px; border-radius: 3px; font-size: 0.9em; }}
        .stat-box {{ display: inline-block; background: #f0f9f9; border: 2px solid #009999; border-radius: 10px; padding: 20px; margin: 10px; text-align: center; min-width: 150px; }}
        .stat-box .number {{ font-size: 2em; font-weight: bold; color: #009999; }}
        .stat-box .label {{ font-size: 0.85em; color: #666; margin-top: 5px; }}
        .method-steps {{ background: #f8fffe; border-left: 4px solid #009999; padding: 15px 20px; margin: 15px 0; }}
        .method-steps ol {{ margin: 5px 0; padding-left: 20px; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>Siemens Electrolyzer Patent Deep Dive</h1>
        <p>Name Harmonisation &amp; Patent Landscape Analysis</p>
        <p>Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} | Methodology: Eurostat/ECOOM (KS-RA-11-008-EN)</p>
    </div>
    <div class="container">

        <div class="section">
            <h2>Key Metrics</h2>
            <div style="text-align: center;">
                <div class="stat-box"><div class="number">{siemens_total}</div><div class="label">Patent Families</div></div>
                <div class="stat-box"><div class="number">{len(harm_df)}</div><div class="label">Name Variations</div></div>
                <div class="stat-box"><div class="number">{harm_df["condensed"].nunique()}</div><div class="label">After Condensing</div></div>
                <div class="stat-box"><div class="number">{harm_df["group"].nunique()}</div><div class="label">Corporate Groups</div></div>
                <div class="stat-box"><div class="number">{(1 - harm_df["condensed"].nunique()/len(harm_df))*100:.0f}%</div><div class="label">Name Reduction</div></div>
                <div class="stat-box"><div class="number">{siemens_total/total_families_all*100:.1f}%</div><div class="label">Share of Total</div></div>
            </div>
        </div>

        <div class="section">
            <h2>Name Harmonisation Methodology</h2>
            <div class="method-steps">
                <p>Following the Eurostat/ECOOM methodology (KS-RA-11-008-EN), each Siemens name variant passes through:</p>
                <ol>
                    <li><b>Character Cleaning</b> - HTML entities, unicode normalisation to ASCII</li>
                    <li><b>Punctuation Cleaning</b> - Normalise spaces, commas, periods</li>
                    <li><b>Legal Form Removal</b> - AG, GmbH, Co. KG, Inc., Ltd., S.A., etc.</li>
                    <li><b>Common Company Word Removal</b> - Company, International, Global, etc.</li>
                    <li><b>Spelling Variation Harmonisation</b> - Systems/Systeme, Technologies/Technologien</li>
                    <li><b>Condensing</b> - Remove all non-alphanumeric characters</li>
                    <li><b>Umlaut Harmonisation</b> - ae/oe/ue variants</li>
                    <li><b>Group Assignment</b> - Map condensed names to corporate groups</li>
                </ol>
            </div>
        </div>

        <div class="section">
            <h2>Name Harmonisation Flow</h2>
            <div class="chart-container">{chart_divs["sankey"]}</div>
        </div>

        <div class="section">
            <h2>All Name Variants (Harmonisation Detail)</h2>
            <table>
                <thead><tr><th>Original Name (PATSTAT)</th><th>After Legal Form Removal</th><th>Condensed</th><th>Corporate Group</th><th>Families</th></tr></thead>
                <tbody>{harm_table_rows}</tbody>
            </table>
        </div>

        <div class="section">
            <h2>Timeline: Patent Activity by Corporate Group</h2>
            <div class="chart-container">{chart_divs["timeline"]}</div>
        </div>

        <div class="section">
            <h2>Technology Profile (IPC)</h2>
            <div class="chart-container">{chart_divs["ipc"]}</div>
            <table>
                <thead><tr><th>IPC Class</th><th>Families</th></tr></thead>
                <tbody>{ipc_table_rows}</tbody>
            </table>
        </div>

        <div class="section">
            <h2>Geographic Filing Strategy</h2>
            <div class="chart-container">{chart_divs["geo"]}</div>
        </div>

    </div>
</body>
</html>"""

# Embed plotly.js inline
import plotly as _pl
with open(os.path.join(os.path.dirname(_pl.__file__), "package_data", "plotly.min.js")) as _f:
    html_content = html_content.replace("__PLOTLY_JS_PLACEHOLDER__", _f.read())

with open(html_filename, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f'HTML report created: {html_filename}')
print(f'   Size: {os.path.getsize(html_filename) / 1024 / 1024:.1f} MB')
print('   Open via: Right-click -> Open in New Browser Tab')


HTML report created: Siemens_Electrolyzer_Deep_Dive.html
   Size: 4.5 MB
   Open via: Right-click -> Open in New Browser Tab


In [14]:
# ==============================================
# DATA EXPORTS
# ==============================================

print("Creating export files...")

csv_filename = "Siemens_Electrolyzer_Patent_Data.csv"
export_cols = ['Applicant_Name', 'Harmonised_Name', 'Siemens_Group', 'Patent_Family_ID', 'Filing_Year', 'Person_ID', 'PSN_Sector', 'Sector']
siemens_raw[export_cols].to_csv(csv_filename, index=False)
print(f"CSV: {csv_filename} ({len(siemens_raw)} rows)")

excel_filename = "Siemens_Electrolyzer_Analysis.xlsx"
with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    harm_export = harm_df[['original', 'char_cleaned', 'legal_removed', 'common_removed', 'condensed', 'group', 'patent_families']]
    harm_export.columns = ['Original_Name', 'Char_Cleaned', 'Legal_Removed', 'Common_Words_Removed', 'Condensed', 'Corporate_Group', 'Patent_Families']
    harm_export.sort_values(['Corporate_Group', 'Patent_Families'], ascending=[True, False]).to_excel(
        writer, sheet_name='Name_Harmonisation', index=False)
    
    group_summary = siemens_raw.groupby('Siemens_Group').agg(
        Name_Variations=('Applicant_Name', 'nunique'),
        Patent_Families=('Patent_Family_ID', 'nunique'),
        First_Year=('Filing_Year', 'min'),
        Last_Year=('Filing_Year', 'max'),
    ).sort_values('Patent_Families', ascending=False).reset_index()
    group_summary.to_excel(writer, sheet_name='Corporate_Groups', index=False)
    
    ipc_summary = ipc_df.groupby('IPC_Class')['Family_ID'].nunique().sort_values(ascending=False).reset_index()
    ipc_summary.columns = ['IPC_Class', 'Patent_Families']
    ipc_summary.to_excel(writer, sheet_name='IPC_Profile', index=False)
    
    geo_summary = publn_df.groupby('Publn_Auth')['Family_ID'].nunique().sort_values(ascending=False).reset_index()
    geo_summary.columns = ['Patent_Office', 'Patent_Families']
    geo_summary.to_excel(writer, sheet_name='Filing_Offices', index=False)
    
    siemens_raw[export_cols].to_excel(writer, sheet_name='Raw_Data', index=False)

print(f"Excel: {excel_filename}")
print(f"\nOutput Files:")
print(f"   Siemens_Electrolyzer_Deep_Dive.html")
print(f"   {csv_filename}")
print(f"   {excel_filename}")


Creating export files...
CSV: Siemens_Electrolyzer_Patent_Data.csv (442 rows)
Excel: Siemens_Electrolyzer_Analysis.xlsx

Output Files:
   Siemens_Electrolyzer_Deep_Dive.html
   Siemens_Electrolyzer_Patent_Data.csv
   Siemens_Electrolyzer_Analysis.xlsx


In [15]:
# ==============================================
# COMPLETE
# ==============================================

print("=" * 80)
print("SIEMENS ELECTROLYZER DEEP DIVE - COMPLETE")
print("=" * 80)
print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Patent families (total):    {siemens_total}")
print(f"  Name variants (original):   {len(harm_df)}")
print(f"  After condensing:           {harm_df['condensed'].nunique()}")
print(f"  Corporate groups:           {harm_df['group'].nunique()}")
print(f"  Name reduction:             {(1 - harm_df['condensed'].nunique()/len(harm_df))*100:.1f}%")
print(f"  Share of electrolyzer pat.: {siemens_total/total_families_all*100:.1f}%")


SIEMENS ELECTROLYZER DEEP DIVE - COMPLETE
Completed: 2026-03-11 08:40:01
  Patent families (total):    97
  Name variants (original):   18
  After condensing:           11
  Corporate groups:           6
  Name reduction:             38.9%
  Share of electrolyzer pat.: 0.4%
